<a href="https://colab.research.google.com/github/BiagioLuc/Visual-Place-Recognition-Project/blob/main/Distribution_of_the_number_of_inliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

#From the file z_data.torch we take the first retrievied image and the positive queries
z_data_path = "/content/drive/MyDrive/Cosplace/logsProgetto/logs_Cosplace_sanF/log_dir/2026-05-15_08-38-15/z_data.torch"
preds_matcher_dir = "/content/drive/MyDrive/Cosplace/logsProgetto/logs_Cosplace_sanF/log_dir/2026-05-15_08-38-15/preds_superpoint-lg"

z_data = torch.load(z_data_path, weights_only=False)
predictions = z_data['predictions']
positives_per_query = z_data['positives_per_query']

num_queries = len(predictions)

#We verify if the prediction is in the positive queries or not
is_r1_correct = []
for i in range(num_queries):
    top1_pred = predictions[i][0].item()
    gt_positives = positives_per_query[i]

    if top1_pred in gt_positives:
        is_r1_correct.append(True)
    else:
        is_r1_correct.append(False)

inliers_correct_queries = []
inliers_wrong_queries = []
file_non_trovati = 0

for q_idx in range(num_queries):
    file_name = f"{q_idx:03d}.torch"
    file_path = os.path.join(preds_matcher_dir, file_name)

    if not os.path.exists(file_path):
        file_non_trovati += 1
        continue

    matching_results = torch.load(file_path, weights_only=False)

    #Taking the number of inliers associate at the first retrieved image
    if len(matching_results) > 0:
        first_match = matching_results[0]
        num_inliers = int(first_match['num_inliers'])

        if is_r1_correct[q_idx]:
            inliers_correct_queries.append(num_inliers)
        else:
            inliers_wrong_queries.append(num_inliers)

all_data = inliers_correct_queries + inliers_wrong_queries
plt.figure(figsize=(10, 6))
max_inliers = max(all_data) if max(all_data) > 0 else 1
# We use discrete bin
bins = np.arange(0, max_inliers + 10, 10)

#Plot distributions
plt.hist(inliers_correct_queries, bins=bins, alpha=0.6, label='Correct Queries (R@1 True)', color='green', edgecolor='black')
plt.hist(inliers_wrong_queries, bins=bins, alpha=0.6, label='Wrong Queries (R@1 False)', color='red', edgecolor='black')

metodo_name = Path(preds_matcher_dir).name
plt.title(f'Distribution of the number of inliers  with recall (R@1) - {metodo_name}')
plt.xlabel('Number of inliers')
plt.ylabel('Number of queries')
plt.legend(loc='upper right', title='Bin width: 10')
plt.grid(axis='y', linestyle='--', alpha=0.7)

output_plot_name = f"correlazione_inliers_{metodo_name}.png"
plt.savefig(output_plot_name)
try:
    plt.savefig(f"/content/drive/MyDrive/Cosplace/logsProgetto/logs_Cosplace_sanF/log_dir/2026-05-15_08-38-15/{output_plot_name}")
except:
    pass
plt.show()